In [6]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[1] # go up n levels (adjust as needed)
sys.path.append(str(ROOT))

from config import PROJECT_ROOT, APT_ROOT
from apt_project import *

In [21]:
import pandas as pd
import numpy as np
import pycountry
import plotly.express as px
import geopandas as gpd
import matplotlib.pyplot as plt

In [26]:
# Turn a country name into its iso3
def to_iso3(country):
    if pd.isna(country):
        return None
    try:
        return pycountry.countries.lookup(country).alpha_3
    except:
        return None

umd_df["target_iso3"] = umd_df["country"].apply(to_iso3)
umd_df["origin_iso3"] = umd_df["actor_country"].apply(to_iso3)

def iso3_to_name(code):
    try:
        return pycountry.countries.get(alpha_3=code).name
    except:
        return code   # fallback

In [33]:
target_country_counts = (
    umd_df.groupby("target_iso3")
          .size()
          .reset_index(name="num_events")
)
target_country_counts = target_country_counts.sort_values("num_events", ascending=False)
target_country_counts["log_events"] = np.log(target_country_counts["num_events"])
target_country_counts["country_name"] = target_country_counts["target_iso3"].apply(iso3_to_name)

origin_country_counts = (
    umd_df.groupby("origin_iso3")
          .size()
          .reset_index(name="num_events")
)
origin_country_counts = origin_country_counts.sort_values("num_events", ascending=False)
origin_country_counts["log_events"] = np.log(origin_country_counts["num_events"])
origin_country_counts["country_name"] = origin_country_counts["origin_iso3"].apply(iso3_to_name)

In [32]:
target_country_counts

,target_iso3,num_events,log_events,country_name
147,USA,7561,3.878579,United States
47,GBR,830,2.919078,United Kingdom
121,RUS,473,2.674861,Russian Federation
67,ITA,459,2.661813,Italy
145,UKR,430,2.633468,Ukraine
...,...,...,...,...
124,SDN,1,0.000000,Sudan
141,TUN,1,0.000000,Tunisia
151,VUT,1,0.000000,Vanuatu
152,YEM,1,0.000000,Yemen


In [34]:
fig = px.choropleth(
    target_country_counts,
    locations="target_iso3",
    color="log_events",
    hover_name="country_name",
    hover_data={
        "num_events": True,          # show real counts
        "log_events": False,         # hide the log value
        "target_iso3": False         # hide if you don't need it
    },
    color_continuous_scale="Reds",
    projection="natural earth",
    title="UMD Cyber Events Heat Map",
)

fig.update_layout(
    coloraxis_colorbar=dict(
        title="Number of Events",
    )
)

fig.show()

In [36]:
N = 10  # change this to whatever number you want

top_countries = (
    origin_country_counts
    .sort_values("num_events", ascending=False)
    .head(N)
)

fig = px.bar(
    top_countries,
    x="num_events",
    y="country_name",     # readable names
    orientation="h",      # horizontal bars
    title=f"Top {N} Countries by Number of Cyber Events",
    labels={"num_events": "Number of Events", "country_name": "Country"},
)

fig.update_layout(
    yaxis=dict(categoryorder="total ascending"),
    xaxis_title="Event Count",
    yaxis_title="",
)

fig.show()